In [ ]:
import pandas as pd
import numpy as np
import time
from decimal import Decimal, getcontext
import os

print("Libraries Loaded")

df = pd.read_csv("/scratch/zt1/project/areilly2-prj/user/mrenna/Part_Two_245_NFIP_Shell.csv")

print("Shell Loaded")

df = df.sort_values(['SBL', 'Year'])

def compute_npv_vectorized(window_df: pd.DataFrame, suffix: str) -> tuple[float, float]:

    def _col_values(col_name: str, fallback: str | int | float = 0) -> np.ndarray:
        series = window_df.get(col_name)
        if series is None and isinstance(fallback, str):
            series = window_df.get(fallback, 0)
        if series is None:
            return np.zeros(len(window_df))
        return series.fillna(0).astype(float).to_numpy()

    market_vals = _col_values(f"Market_Project_{suffix}", "Market_Project")
    dam_none = _col_values(f"Dam_T_{suffix}", "Dam_T")
    dam_nfip_deduct = _col_values(f"Dam_NFIP_Deduct_{suffix}", "Dam_NFIP_Deduct")
    dam_nfip_private_deduct = _col_values(
        f"Dam_NFIP_Private_Deduct_{suffix}", "Dam_NFIP_Private_Deduct"
    )
    premium_nfip = _col_values("NFIP_Premium")
    premium_private = _col_values("NFIP_Private_Premium")
    replace_costs = _col_values("ReplaceCost")

    t_arr = np.arange(1, len(window_df) + 1)
    discount = np.power(1.03, t_arr)

    mask_nonzero = replace_costs != 0
    exp_none = np.zeros_like(replace_costs)
    exp_nfip = np.zeros_like(replace_costs)
    exp_none[mask_nonzero] = (
        -1 / (0.05 * replace_costs[mask_nonzero])
    ) * (market_vals[mask_nonzero] - dam_none[mask_nonzero])
    exp_nfip[mask_nonzero] = (
        -1 / (0.05 * replace_costs[mask_nonzero])
    ) * (
        market_vals[mask_nonzero]
        - dam_nfip_deduct[mask_nonzero]
        - dam_nfip_private_deduct[mask_nonzero]
        - premium_nfip[mask_nonzero]
        - premium_private[mask_nonzero]
    )

    npv_none_arr = np.where(
        mask_nonzero, (1 / discount) * (1 - np.exp(exp_none)), 0
    )
    npv_nfip_arr = np.where(
        mask_nonzero, (1 / discount) * (1 - np.exp(exp_nfip)), 0
    )
    deu_none = npv_none_arr.sum()
    deu_nfip = npv_nfip_arr.sum()
    return deu_none, deu_nfip
    
getcontext().prec = 15

intensity = ['00000', '00005', '00010', '00030', '00050', '00100', '00300', '00500', '01000', '10000']
weights = {k: 1 / int(k) for k in intensity if k != '00000'}
weights['00000'] = 1 - sum(weights.values())

start_time = time.time()

year_range = list(range(2025, 2076))
bins = [0, 6303, 8303, 9303, 9636, 9836, 9936, 9969, 9989, 9999, 10001]
labels = [0, 5, 10, 30, 50, 100, 300, 500, 1000, 10000]
random_numbers = np.random.randint(0, 10001, size=len(year_range))
events = pd.cut(random_numbers, bins=bins, labels=labels, right=False).astype(int)
event_map = dict(zip(year_range, events))

year_sbls = df.groupby('Year')['SBL'].unique().to_dict()
group_idx = df.groupby(['SBL', 'Year']).groups
year_idx = df.groupby('Year').groups
sbl_idx = df.groupby('SBL').groups

print("Begin Calculating")

for year in year_range:  
    year_start = time.time()
    df.loc[year_idx.get(year, []), 'Flood_Event'] = event_map[year]
    
    for sbl in year_sbls.get(year, []):
        sbl_start = time.time()
        print(f"\nProcessing SBL: {sbl}")
        
        idx = group_idx.get((sbl, year))
        base_row = df.loc[idx] if idx is not None else pd.DataFrame()

        if base_row.empty:
            continue
            
        occupy = base_row['Occupy_NFIP'].values[0]

        if occupy == 0:
            df.loc[idx, [
                'Dam_T', 'Dam_T_Con', 'Dam_T_Struc',
                'Dam_NFIP', 'Dam_NFIP_Con', 'Dam_NFIP_Struc', 'Dam_NFIP_Deduct',
                'Dam_NFIP_Private', 'Dam_NFIP_Private_Deduct', 'Dam_None',
                'NFIP_Premium_Paid', 'NFIP_Private_Premium_Paid',
                'Decision_NFIP', 'Decision_None'
            ]] = 0
            continue
            
        flood_occur_base_factors = [
            df.loc[idx, f'Flood_Occur_{i}_Factor'].fillna(1).values[0]
            for i in range(1, 5)
        ]
        flood_occur_min = min(flood_occur_base_factors)
        
        prox_flood_base_factors = [
            df.loc[idx, f'Prox_Flood_{i}_Factor'].fillna(1).values[0]
            for i in range(1, 5)
        ]
        prox_flood_min = min(prox_flood_base_factors)

        deu_none = 0
        deu_nfip = 0

        slr_year = df.loc[sbl_idx.get(sbl, []), 'SLR'].dropna().values[0]
        end_year = min(year + 30, slr_year + 1) 

        for suffix in intensity:
            npv_values_none = []
            npv_values_nfip = []

            if suffix == '00000':
                
                mask_y = (df['Year'].between(year, end_year - 1)) & (df['SBL'] == sbl)
                df.loc[mask_y, ['Flood_Occur_00000_Factor', 'Prox_Flood_00000_Factor']] = [flood_occur_min, prox_flood_min]
                df.loc[mask_y, 'Flood_Factor_00000'] = np.minimum(
                    df.loc[mask_y, 'Flood_Occur_00000_Factor'], df.loc[mask_y, 'Prox_Flood_00000_Factor']
                )
            
            else:
                col_name = f'Flood_Occur_0_{suffix}_Factor'
                
                if col_name in df.columns:
                    
                    mask_y = (df['Year'].between(year, end_year - 1)) & (df['SBL'] == sbl)
                    vals = df.loc[mask_y, col_name].fillna(1)
                    flood_occur_vals = np.minimum(flood_occur_min, vals)
                    prox_col = f'Prox_Flood_0_{suffix}_Factor'
                    prox_vals = df.loc[mask_y, prox_col].fillna(1)
                    prox_flood_vals = np.minimum(prox_flood_min, prox_vals)
                    df.loc[mask_y, [f'Flood_Occur_{suffix}_Factor', f'Prox_Flood_{suffix}_Factor']] = np.vstack([flood_occur_vals, prox_flood_vals]).T
                    df.loc[mask_y, f'Flood_Factor_{suffix}'] = np.minimum(flood_occur_vals, prox_flood_vals)

            for y in range(year, end_year):
                idx_y = group_idx.get((sbl, y))
                if idx_y is None:
                    continue
                flood_factor = df.loc[idx_y, f'Flood_Factor_{suffix}'].fillna(1).values[0]
                initial_val = df.loc[idx_y, 'Initial_Market_Value'].fillna(1).values[0]
                external_val = df.loc[idx_y, 'External_Factor'].fillna(1).values[0]
                slr_val = df.loc[idx_y, 'SLR_Factor'].fillna(1).values[0]
                df.loc[idx_y, f'Market_Project_{suffix}'] = (
                    initial_val * external_val * slr_val * flood_factor
                )
                
            window_df = df.loc[sbl_idx.get(sbl, []) , :]
            window_df = window_df[(window_df["Year"] >= year) & (window_df["Year"] < end_year)]

            deu_none_suffix, deu_nfip_suffix = compute_npv_vectorized(window_df, suffix)

            weight = weights.get(suffix, 0)
            deu_none += weight * deu_none_suffix
            deu_nfip += weight * deu_nfip_suffix

        df.loc[idx, 'DEU_None'] = deu_none
        df.loc[idx, 'DEU_NFIP'] = deu_nfip
        
        print(f"SBL {sbl}, Year {year}: DEU_None = {deu_none}, DEU_NFIP = {deu_nfip}")

        nfip_ = df.loc[idx, 'NFIP_Premium'].fillna(0).values[0]
        nfip_private = df.loc[idx, 'NFIP_Private_Premium'].fillna(0).values[0]

        if deu_none >= deu_nfip:  
            df.loc[idx, 'Decision_None'] = 1
            df.loc[idx, 'Decision_NFIP'] = 0
        else:
            df.loc[idx, 'Decision_None'] = 0
            df.loc[idx, 'Decision_NFIP'] = 1

        zero_idx = [i for i in group_idx.get((sbl, year), []) if df.loc[i, 'Flood_Event'] == 0]
                                    
        df.loc[zero_idx, 'Dam_T_Con'] = 0
        df.loc[zero_idx, 'Dam_T_Struc'] = 0
        df.loc[zero_idx, 'Dam_T'] = 0,
        df.loc[zero_idx, 'Flood_Occur_0_Factor'] = 1
        df.loc[zero_idx, 'Flood_Occur_Factor'] = df.loc[zero_idx, 'Flood_Occur_00000_Factor']
        df.loc[zero_idx, 'Prox_Flood_0_Factor'] = 1
        df.loc[zero_idx, 'Prox_Flood_Factor'] = df.loc[zero_idx, 'Prox_Flood_00000_Factor']
        df.loc[zero_idx, 'Flood_Factor'] = df.loc[zero_idx, 'Flood_Factor_00000']
        df.loc[zero_idx, 'Market_Value_Avg_NFIP'] = df.loc[zero_idx, 'Market_Project_00000']
        df.loc[zero_idx, 'Dam_NFIP_Con'] = 0
        df.loc[zero_idx, 'Dam_NFIP_Struc'] = 0
        df.loc[zero_idx, 'Dam_NFIP'] = 0
        df.loc[zero_idx, 'Dam_NFIP_Deduct'] = 0
        df.loc[zero_idx, 'Dam_NFIP_Private'] = 0
        df.loc[zero_idx, 'Dam_NFIP_Private_Deduct'] = 0
        df.loc[zero_idx, 'Dam_None'] = 0

        nfip_zero_idx = [i for i in zero_idx if df.loc[i, 'Decision_NFIP'] == 1]        
        none_zero_idx = [i for i in zero_idx if df.loc[i, 'Decision_None'] == 1]
        
        df.loc[nfip_zero_idx, 'NFIP_Premium_Paid'] = df.loc[nfip_zero_idx, 'NFIP_Premium']        
        df.loc[nfip_zero_idx, 'NFIP_Private_Premium_Paid'] = df.loc[nfip_zero_idx, 'NFIP_Private_Premium']        
        df.loc[none_zero_idx, 'NFIP_Premium_Paid'] = 0
        df.loc[none_zero_idx, 'NFIP_Private_Premium_Paid'] = 0

        
        mask = df.index.isin(idx)
        if df.loc[mask, 'SLR'].values[0] == year:
            replace_cost = df.loc[mask, 'ReplaceCost'].values[0]

            df.loc[mask, 'Dam_T_Con'] = 0.5 * replace_cost
            df.loc[mask, 'Dam_T_Struc'] = replace_cost
            df.loc[mask, 'Dam_T'] = 1.5 * replace_cost
            df.loc[flood_event_mask, 'Market_Value_Avg_NFIP'] = 0

            nfip_mask = mask & (df['Decision_NFIP'] == 1)
            if nfip_mask.any():
                dam_t_con = df.loc[nfip_mask, 'Dam_T_Con'].values[0]
                dam_t_struc = df.loc[nfip_mask, 'Dam_T_Struc'].values[0]
                dam_t = dam_t_con + dam_t_struc

                dam_nfip_con = 100000 if dam_t_con >= 111111.11 else 0.9 * dam_t_con
                dam_nfip_struc = 250000 if dam_t_struc >= 277777.78 else 0.9 * dam_t_struc
                dam_nfip = dam_nfip_con + dam_nfip_struc
                dam_nfip_deduct = dam_nfip * (1 / 9)
                dam_nfip_private = 0.9 * (dam_t - dam_nfip - dam_nfip_deduct)
                dam_nfip_private_deduct = (1 / 9) * dam_nfip_private

                df.loc[nfip_mask, 'Dam_NFIP_Con'] = dam_nfip_con
                df.loc[nfip_mask, 'Dam_NFIP_Struc'] = dam_nfip_struc
                df.loc[nfip_mask, 'Dam_NFIP'] = dam_nfip
                df.loc[nfip_mask, 'Dam_NFIP_Deduct'] = dam_nfip_deduct
                df.loc[nfip_mask, 'Dam_NFIP_Private'] = dam_nfip_private
                df.loc[nfip_mask, 'Dam_NFIP_Private_Deduct'] = dam_nfip_private_deduct
                df.loc[nfip_mask, 'Dam_None'] = 0
                df.loc[nfip_mask, 'NFIP_Premium_Paid'] = df.loc[nfip_mask, 'NFIP_Premium']
                df.loc[nfip_mask, 'NFIP_Private_Premium_Paid'] = df.loc[nfip_mask, 'NFIP_Private_Premium']

            none_mask = mask & (df['Decision_None'] == 1)
            if none_mask.any():
                df.loc[none_mask, 'Dam_NFIP_Con'] = 0
                df.loc[none_mask, 'Dam_NFIP_Struc'] = 0
                df.loc[none_mask, 'Dam_NFIP'] = 0
                df.loc[none_mask, 'Dam_NFIP_Deduct'] = 0
                df.loc[none_mask, 'Dam_NFIP_Private'] = 0
                df.loc[none_mask, 'Dam_NFIP_Private_Deduct'] = 0
                df.loc[none_mask, 'Dam_None'] = df.loc[none_mask, 'Dam_T']
                df.loc[none_mask, 'NFIP_Premium_Paid'] = 0
                df.loc[none_mask, 'NFIP_Private_Premium_Paid'] = 0

        
        else:
            for suffix in intensity:
                if suffix == '00000':
                    continue
                event_val = int(suffix)
                flood_event_mask = mask & (df['Flood_Event'] == event_val)

                df.loc[flood_event_mask, 'Dam_T_Con'] = df.loc[flood_event_mask, f'Dam_T_Con_{suffix}']
                df.loc[flood_event_mask, 'Dam_T_Struc'] = df.loc[flood_event_mask, f'Dam_T_Struc_{suffix}']
                df.loc[flood_event_mask, 'Dam_T'] = df.loc[flood_event_mask, f'Dam_T_{suffix}']
                df.loc[flood_event_mask, 'Flood_Occur_0_Factor'] = df.loc[flood_event_mask, f'Flood_Occur_0_{suffix}_Factor']
                df.loc[flood_event_mask, 'Flood_Occur_Factor'] = df.loc[flood_event_mask, f'Flood_Occur_{suffix}_Factor']
                df.loc[flood_event_mask, 'Prox_Flood_0_Factor'] = df.loc[flood_event_mask, f'Prox_Flood_0_{suffix}_Factor']
                df.loc[flood_event_mask, 'Prox_Flood_Factor'] = df.loc[flood_event_mask, f'Prox_Flood_{suffix}_Factor']
                df.loc[flood_event_mask, 'Flood_Factor'] = df.loc[flood_event_mask, f'Flood_Factor_{suffix}']
                df.loc[flood_event_mask, 'Market_Value_Avg_NFIP'] = df.loc[flood_event_mask, f'Market_Project_{suffix}']

                nfip_mask = flood_event_mask & (df['Decision_NFIP'] == 1)
                df.loc[nfip_mask, 'Dam_NFIP_Con'] = df.loc[nfip_mask, f'Dam_NFIP_Con_{suffix}']
                df.loc[nfip_mask, 'Dam_NFIP_Struc'] = df.loc[nfip_mask, f'Dam_NFIP_Struc_{suffix}']
                df.loc[nfip_mask, 'Dam_NFIP'] = df.loc[nfip_mask, f'Dam_NFIP_{suffix}']
                df.loc[nfip_mask, 'Dam_NFIP_Deduct'] = df.loc[nfip_mask, f'Dam_NFIP_Deduct_{suffix}']
                df.loc[nfip_mask, 'Dam_NFIP_Private'] = df.loc[nfip_mask, f'Dam_NFIP_Private_{suffix}']
                df.loc[nfip_mask, 'Dam_NFIP_Private_Deduct'] = df.loc[nfip_mask, f'Dam_NFIP_Private_Deduct_{suffix}']
                df.loc[nfip_mask, 'Dam_None'] = 0
                df.loc[nfip_mask, 'NFIP_Premium_Paid'] = df.loc[nfip_mask, 'NFIP_Premium']
                df.loc[nfip_mask, 'NFIP_Private_Premium_Paid'] = df.loc[nfip_mask, 'NFIP_Private_Premium']

                none_mask = flood_event_mask & (df['Decision_None'] == 1)
                df.loc[none_mask, 'Dam_NFIP_Con'] = 0
                df.loc[none_mask, 'Dam_NFIP_Struc'] = 0
                df.loc[none_mask, 'Dam_NFIP'] = 0
                df.loc[none_mask, 'Dam_NFIP_Deduct'] = 0
                df.loc[none_mask, 'Dam_NFIP_Private'] = 0
                df.loc[none_mask, 'Dam_NFIP_Private_Deduct'] = 0
                df.loc[none_mask, 'Dam_None'] = df.loc[none_mask, 'Dam_T']
                df.loc[none_mask, 'NFIP_Premium_Paid'] = 0
                df.loc[none_mask, 'NFIP_Private_Premium_Paid'] = 0
        
        next_year = year + 1
        mask_next = (df['SBL'] == sbl) & (df['Year'] == next_year)
        mask_current = (df['SBL'] == sbl) & (df['Year'] == year)

        flood_0 = df.loc[mask_current, 'Flood_Occur_0_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Flood_Occur_1_Factor'] = np.where(flood_0 < 1, 0.844, 1)

        flood_1 = df.loc[mask_current, 'Flood_Occur_1_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Flood_Occur_2_Factor'] = np.where(flood_1 < 1, 0.883, 1)

        flood_2 = df.loc[mask_current, 'Flood_Occur_2_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Flood_Occur_3_Factor'] = np.where(flood_2 < 1, 0.922, 1)

        flood_3 = df.loc[mask_current, 'Flood_Occur_3_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Flood_Occur_4_Factor'] = np.where(flood_3 < 1, 0.961, 1)

        prox_0 = df.loc[mask_current, 'Prox_Flood_0_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Prox_Flood_1_Factor'] = np.where(prox_0 < 1, 0.948, 1)

        prox_1 = df.loc[mask_current, 'Prox_Flood_1_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Prox_Flood_2_Factor'] = np.where(prox_1 < 1, 0.961, 1)

        prox_2 = df.loc[mask_current, 'Prox_Flood_2_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Prox_Flood_3_Factor'] = np.where(prox_2 < 1, 0.974, 1)

        prox_3 = df.loc[mask_current, 'Prox_Flood_3_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Prox_Flood_4_Factor'] = np.where(prox_3 < 1, 0.987, 1) 

        sbl_duration = time.time() - sbl_start
        print(f"Finished SBL: {sbl} in {sbl_duration:.2f} seconds")
        
    df['Cost_NFIP'] = df.groupby('SBL')['Dam_NFIP'].transform('cumsum')
    df['Cost_NFIP_Deduct'] = df.groupby('SBL')['Dam_NFIP_Deduct'].transform('cumsum')
    df['Cost_NFIP_Private'] = df.groupby('SBL')['Dam_NFIP_Private'].transform('cumsum')
    df['Cost_NFIP_Private_Deduct'] = df.groupby('SBL')['Dam_NFIP_Private_Deduct'].transform('cumsum')
    df['Cost_None'] = df.groupby('SBL')['Dam_None'].transform('cumsum')
    df['Cost_NFIP_Premium'] = df.groupby('SBL')['NFIP_Premium_Paid'].transform('cumsum')
    df['Cost_NFIP_Private_Premium'] = df.groupby('SBL')['NFIP_Private_Premium_Paid'].transform('cumsum')

    year_duration = time.time() - year_start
    print(f"Year {year} processed in {year_duration:.2f} seconds")

total_duration = time.time() - start_time
print(f"\nTotal time for all SBLs and years: {total_duration:.2f} seconds")

columns_to_keep = [
    'SBL',
    'Year',
    'Occupy_NFIP',
    'Market_Value_Avg_NFIP',
    'Decision_NFIP',
    'Decision_None',
    'Dam_T',
    'Cost_NFIP',
    'Cost_NFIP_Deduct',
    'Cost_NFIP_Private',
    'Cost_NFIP_Private_Deduct',
    'Cost_NFIP_Premium',
    'Cost_NFIP_Private_Premium',
    'Cost_None'
]

df_final = df.loc[
    df['Year'].between(2025, 2075),
    columns_to_keep
]

slurm_job_id = os.getenv("SLURM_JOB_ID", "manual")
output_filename = f"Part_Two_NFIP_245_Simulation_{slurm_job_id}.csv"
output_path = f"/scratch/zt1/project/areilly2-prj/user/mrenna/{output_filename}"

print(f"Saving output to: {output_path}")
df_final.to_csv(output_path, index=False)